# L11：RNN与序列数据处理


# L11：RNN与序列数据处理

## 学习目标

1. 理解序列数据的特点与RNN的引入动机
2. 掌握RNN的前向传播与反向传播（BPTT）原理
3. 理解长短期记忆网络（LSTM）解决梯度消失的机制
4. 掌握门控循环单元（GRU）的结构与原理
5. 能够实现序列到序列（Seq2Seq）的翻译模型

## 核心知识点

### 1. 序列数据的特殊性与RNN的引入

序列数据无处不在：文本、语音、时间序列、视频帧序列等。这类数据的特点是：
- **顺序依赖**：第 t 步的输出可能依赖于之前所有步骤的信息
- **可变长度**：不同样本的长度可以不同
- **位置编码重要**：元素的相对位置影响语义

传统全连接网络无法有效处理这种"可变长度且有顺序依赖"的数据，这促使了循环神经网络（RNN）的诞生。

### 2. RNN的基本结构

RNN的核心思想是引入"隐藏状态"（hidden state）来记忆历史信息：


In [ ]:
h_t = f(W_xh * x_t + W_hh * h_{t-1} + b)
y_t = g(W_hy * h_t + c)


其中 h_t 是时刻 t 的隐藏状态，f 通常是 tanh 或 ReLU，g 是输出层的激活函数。

### 3. RNN的前向传播

给定输入序列 x = (x_1, x_2, ..., x_T)，RNN的计算过程：


In [ ]:
h_0 = zeros
for t in 1..T:
    h_t = tanh(W_xh * x_t + W_hh * h_{t-1} + b)
    y_t = W_hy * h_t + c


时间复杂度：O(T)，其中T为序列长度。

### 4. 通过时间反向传播（BPTT）

RNN的反向传播需要沿时间轴展开，因此称为BPTT（Backpropagation Through Time）。

损失函数定义为各时间步损失之和：L = Σ_t L_t

对权重 W_xh 的梯度：

$$\frac{\partial L}{\partial W_{xh}} = \sum_{t=1}^{T} \frac{\partial L_t}{\partial W_{xh}}$$

展开后涉及大量雅可比矩阵的乘积，当序列很长时（t >> 1），容易出现梯度爆炸或梯度消失。

### 5. 梯度消失与梯度爆炸

梯度消失的原因：
- tanh 的导数最大为1
- 沿时间反向传播时，每一步都乘以 W_hh^T
- 当时间步很多时，梯度会指数级衰减

梯度爆炸的迹象：损失突然变成 NaN，或者权重出现非常大的值。

解决方案：
- **梯度裁剪**：将梯度限制在某个范围内
- **LSTM/GRU**：通过门控机制缓解
- **残差连接**：允许梯度直接传递

### 6. 长短期记忆网络（LSTM）

LSTM 由 Hochreiter & Schmidhuber 于1997年提出，是目前应用最广泛的序列模型之一。

LSTM 引入三个门控：
- **遗忘门（Forget Gate）**：决定哪些信息应该被丢弃
- **输入门（Input Gate）**：决定哪些新信息可以存储
- **输出门（Output Gate）**：决定输出什么信息

LSTM 的核心公式：


In [ ]:
f_t = σ(W_f · [h_{t-1}, x_t] + b_f)      # 遗忘门
i_t = σ(W_i · [h_{t-1}, x_t] + b_i)      # 输入门
C̃_t = tanh(W_C · [h_{t-1}, x_t] + b_C)  # 候选记忆
C_t = f_t * C_{t-1} + i_t * C̃_t         # 新记忆
o_t = σ(W_o · [h_{t-1}, x_t] + b_o)      # 输出门
h_t = o_t * tanh(C_t)                     # 新隐藏状态


### 7. 门控循环单元（GRU）

GRU 是 LSTM 的简化版本，由 Cho 等人于2014年提出，只有两个门：


In [ ]:
z_t = σ(W_z · [h_{t-1}, x_t])           # 更新门
r_t = σ(W_r · [h_{t-1}, x_t])           # 重置门
h̃_t = tanh(W · [r_t * h_{t-1}, x_t])   # 候选隐藏状态
h_t = (1 - z_t) * h_{t-1} + z_t * h̃_t  # 新隐藏状态


### 8. 序列到序列模型（Seq2Seq）

Seq2Seq 是机器翻译等任务的标准架构：


In [ ]:
Encoder: (x_1, x_2, ..., x_T) -> c（上下文向量）
Decoder: (y_1, ..., y_{T'})，每个时刻的输入是上一步的输出


注意力机制（Attention）解决了长序列的瓶颈问题：


In [ ]:
attention_score = score(h_t, s_{t-1})
attention_weights = softmax(attention_scores)
context = Σ attention_weights * encoder_hidden_states


### 9. 序列模型中的常用技术

- **词嵌入（Word Embedding）**：将单词映射到稠密向量空间
- **双向RNN**：同时利用前后文信息
- **堆叠RNN**：多层RNN增加模型容量
- ** Teacher Forcing**：训练时使用真实上一步输出而非模型预测

## 代码示例

### 从零实现LSTM


In [ ]:
import numpy as np

class LSTMCell:
    """LSTM单元的实现"""

    def __init__(self, input_size, hidden_size):
        """
        参数：
            input_size: 输入特征的维度
            hidden_size: 隐藏状态的维度
        """
        self.input_size = input_size
        self.hidden_size = hidden_size

        # 合并权重矩阵以提高效率（4个门）
        # W_f, W_i, W_C, W_o 合并为 W
        # 将 x_t 和 h_{t-1} 拼接，维度为 input_size + hidden_size
        combined_size = input_size + hidden_size

        # Xavier初始化
        scale = np.sqrt(2.0 / combined_size)
        self.W = np.random.randn(4 * hidden_size, combined_size) * scale
        self.b = np.zeros((4 * hidden_size, 1))

        # 用于反向传播的缓存
        self.cache = None

    def sigmoid(self, x):
        return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))

    def forward(self, x, h_prev, C_prev):
        """
        前向传播
        参数：
            x: 当前输入 (input_size, batch)
            h_prev: 上一时刻的隐藏状态 (hidden_size, batch)
            C_prev: 上一时刻的记忆 (hidden_size, batch)
        返回：
            h: 新的隐藏状态
            C: 新的记忆
        """
        batch_size = x.shape[1]

        # 拼接 x 和 h_prev
        combined = np.vstack([x, h_prev])  # (input_size + hidden_size, batch)

        # 计算四个门的值
        z = self.W @ combined + self.b  # (4*hidden_size, batch)

        # 分割为四个门
        Gates = np.split(z, 4, axis=0)
        f = self.sigmoid(Gates[0])  # 遗忘门
        i = self.sigmoid(Gates[1])  # 输入门
        C_tilde = np.tanh(Gates[2])  # 候选记忆
        o = self.sigmoid(Gates[3])  # 输出门

        # 计算新的记忆和隐藏状态
        C = f * C_prev + i * C_tilde
        h = o * np.tanh(C)

        # 保存缓存用于反向传播
        self.cache = (x, h_prev, C_prev, f, i, C_tilde, o, C)

        return h, C

    def backward(self, dh_next, dC_next):
        """
        反向传播
        参数：
            dh_next: 来自下一时刻的梯度 (hidden_size, batch)
            dC_next: 记忆的梯度 (hidden_size, batch)
        返回：
            dx: 输入的梯度
            dh_prev: 隐藏状态的梯度
            dC_prev: 记忆的梯度
            dWx, dWh, db: 权重梯度
        """
        x, h_prev, C_prev, f, i, C_tilde, o, C = self.cache
        batch_size = x.shape[1]

        # tanh(C) 的梯度
        do = dh_next * np.tanh(C)
        dC = dC_next + dh_next * o * (1 - np.tanh(C) ** 2)

        # 各个门的梯度
        df = dC * C_prev
        di = dC * C_tilde
        dC_tilde = dC * i
        do_sigmoid = do * o * (1 - o)

        # 组合梯度
        dGates = np.vstack([df, di, dC_tilde, do_sigmoid])

        # 权重梯度
        combined = np.vstack([x, h_prev])
        dW = dGates @ combined.T / batch_size
        db = np.sum(dGates, axis=1, keepdims=True) / batch_size

        # 输入梯度
        dcombined = self.W.T @ dGates / batch_size
        dx = dcombined[:self.input_size, :]
        dh_prev = dcombined[self.input_size:, :]

        # C_prev 的梯度
        dC_prev = dC * f

        return dx, dh_prev, dC_prev, dW, db


class LSTM:
    """完整的LSTM模型"""

    def __init__(self, vocab_size, embed_size, hidden_size):
        self.vocab_size = vocab_size
        self.embed_size = embed_size
        self.hidden_size = hidden_size

        # 词嵌入
        self.embedding = np.random.randn(vocab_size, embed_size) * 0.01

        # LSTM细胞
        self.lstm = LSTMCell(embed_size, hidden_size)

        # 输出层权重
        self.Wy = np.random.randn(vocab_size, hidden_size) * np.sqrt(2.0 / hidden_size)
        self.by = np.zeros((vocab_size, 1))

    def softmax(self, x):
        exp_x = np.exp(x - np.max(x, axis=0, keepdims=True))
        return exp_x / np.sum(exp_x, axis=0, keepdims=True)

    def forward(self, sequence, target=None):
        """
        前向传播
        参数：
            sequence: 输入序列 (seq_len, batch)
            target: 目标序列 (seq_len, batch)，用于计算损失
        返回：
            loss: 交叉熵损失
            output: 预测概率 (vocab_size, seq_len * batch)
        """
        seq_len, batch_size = sequence.shape

        # 初始化
        h = np.zeros((self.hidden_size, batch_size))
        C = np.zeros((self.hidden_size, batch_size))

        self.cache = []

        # 存储所有时刻的隐藏状态（用于反向传播）
        self.h_history = [h]
        self.C_history = [C]

        # 前向传播所有时刻
        for t in range(seq_len):
            # 词嵌入
            x = self.embedding[sequence[t]].T  # (embed_size, batch)

            # LSTM前向传播
            h, C = self.lstm.forward(x, h, C)

            self.h_history.append(h)
            self.C_history.append(C)

        # 输出层
        output = self.Wy @ h + self.by  # (vocab_size, batch)
        output = self.softmax(output)

        # 计算损失
        loss = 0
        if target is not None:
            # 展平
            output_flat = output.T.reshape(-1)
            target_flat = target.T.flatten()
            # 交叉熵损失
            output_flat = np.clip(output_flat, 1e-10, 1 - 1e-10)
            loss = -np.sum(target_flat * np.log(output_flat)) / batch_size

        return loss, output

    def backward(self, target):
        """反向传播"""
        seq_len = len(self.h_history) - 1
        batch_size = target.shape[1]

        # 初始化梯度
        dWy = np.zeros_like(self.Wy)
        dby = np.zeros_like(self.by)
        dh_next = np.zeros((self.hidden_size, batch_size))
        dC_next = np.zeros((self.hidden_size, batch_size))

        # 输出层梯度
        output = self.h_history[-1].T  # (batch, hidden)
        dy = output.copy()
        target_flat = target.T.flatten()
        dy[np.arange(batch_size), target_flat] -= 1
        dy = dy.T  # (hidden, batch)

        dWy = dy @ self.h_history[-2].T.T  # 调整维度
        dby = np.sum(dy, axis=1, keepdims=True)

        dh_next = self.Wy.T @ dy

        # 反向传播所有时刻
        for t in reversed(range(seq_len)):
            # 获取缓存
            self.lstm.cache = self.lstm.cache  # 已经在forward中设置

            # 反向传播
            dx, dh_prev, dC_prev, dW, db = self.lstm.backward(dh_next, dC_next)

            dh_next = dh_prev
            dC_next = dC_prev

        return {'Wy': dWy, 'by': dby}


### 使用PyTorch实现机器翻译


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence


class Encoder(nn.Module):
    """序列到序列模型的编码器"""

    def __init__(self, vocab_size, embed_size, hidden_size, num_layers=2, dropout=0.2):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=0)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers,
                           batch_first=True, dropout=dropout if num_layers > 1 else 0,
                           bidirectional=True)
        self.fc = nn.Linear(hidden_size * 2, hidden_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, lengths):
        """
        参数：
            x: (batch, seq_len) 输入序列
            lengths: (batch,) 序列长度
        返回：
            outputs: (batch, seq_len, hidden_size) 所有时刻的输出
            hidden: 最终隐藏状态
        """
        embedded = self.dropout(self.embedding(x))  # (batch, seq_len, embed_size)

        # 打包序列以高效处理变长序列
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        outputs, (hidden, cell) = self.lstm(packed)

        # 解包
        outputs, _ = nn.utils.rnn.pad_packed_sequence(outputs, batch_first=True)

        # 双向LSTM：合并前向和后向隐藏状态
        # hidden: (num_layers * 2, batch, hidden_size)
        # 需要将前向和后向组合
        hidden = self._combine_directions(hidden)
        cell = self._combine_directions(cell)

        # 将隐藏状态映射到单一方向
        outputs = self.fc(outputs)  # (batch, seq_len, hidden_size)

        return outputs, (hidden, cell)

    def _combine_directions(self, hidden):
        """合并双向隐藏状态"""
        batch_size = hidden.size(1)
        hidden_size = hidden.size(2)
        num_layers = hidden.size(0) // 2

        # 重塑为 (num_layers, 2, batch, hidden)
        hidden = hidden.view(num_layers, 2, batch_size, hidden_size)
        # 拼接前向和后向
        combined = torch.cat([hidden[:, 0, :, :], hidden[:, 1, :, :]], dim=2)
        return combined


class Attention(nn.Module):
    """Luong注意力机制"""

    def __init__(self, hidden_size):
        super(Attention, self).__init__()
        self.attn = nn.Linear(hidden_size * 2, hidden_size)
        self.v = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, hidden, encoder_outputs, mask=None):
        """
        参数：
            hidden: (batch, hidden_size) 当前解码器隐藏状态
            encoder_outputs: (batch, seq_len, hidden_size) 编码器输出
            mask: (batch, seq_len) 填充位置的mask
        返回：
            context: (batch, hidden_size) 上下文向量
            attn_weights: (batch, seq_len) 注意力权重
        """
        batch_size = encoder_outputs.size(0)
        seq_len = encoder_outputs.size(1)

        # 重复hidden以匹配encoder_outputs的序列长度
        hidden_repeated = hidden.unsqueeze(1).repeat(1, seq_len, 1)
        # 拼接hidden和encoder_outputs
        combined = torch.cat([hidden_repeated, encoder_outputs], dim=2)

        # 计算能量
        energy = torch.tanh(self.attn(combined))  # (batch, seq_len, hidden_size)
        attention = self.v(energy).squeeze(2)  # (batch, seq_len)

        # 应用mask
        if mask is not None:
            attention = attention.masked_fill(mask == 0, -1e10)

        # Softmax得到注意力权重
        attn_weights = F.softmax(attention, dim=1)

        # 计算上下文向量
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs).squeeze(1)

        return context, attn_weights


class Decoder(nn.Module):
    """带注意力机制的解码器"""

    def __init__(self, vocab_size, embed_size, hidden_size, num_layers=1, dropout=0.2):
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=0)
        self.attention = Attention(hidden_size)
        self.lstm = nn.LSTM(embed_size + hidden_size, hidden_size, num_layers,
                           batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.fc = nn.Linear(hidden_size * 2 + embed_size, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, hidden, cell, encoder_outputs, mask=None):
        """
        参数：
            x: (batch, 1) 当前输入（单个时刻）
            hidden: (num_layers, batch, hidden_size)
            cell: (num_layers, batch, hidden_size)
            encoder_outputs: (batch, seq_len, hidden_size)
            mask: (batch, seq_len)
        返回：
            output: (batch, vocab_size) 预测分布
            hidden: 更新后的隐藏状态
            attn_weights: 注意力权重
        """
        batch_size = x.size(0)

        embedded = self.dropout(self.embedding(x))  # (batch, 1, embed_size)

        # 使用顶层隐藏状态计算注意力
        top_hidden = hidden[-1]  # (batch, hidden_size)
        context, attn_weights = self.attention(top_hidden, encoder_outputs, mask)

        # 拼接embedded和context
        lstm_input = torch.cat([embedded, context.unsqueeze(1)], dim=2)  # (batch, 1, embed_size+hidden)

        # LSTM前向传播
        output, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))

        # 输出层
        output = torch.cat([output.squeeze(1), context, embedded.squeeze(1)], dim=1)
        output = self.fc(output)  # (batch, vocab_size)

        return output, hidden, cell, attn_weights


class Seq2Seq(nn.Module):
    """序列到序列模型"""

    def __init__(self, src_vocab_size, tgt_vocab_size, embed_size=256,
                 hidden_size=512, num_layers=2, dropout=0.2):
        super(Seq2Seq, self).__init__()
        self.encoder = Encoder(src_vocab_size, embed_size, hidden_size, num_layers, dropout)
        self.decoder = Decoder(tgt_vocab_size, embed_size, hidden_size, num_layers, dropout)
        self.tgt_vocab_size = tgt_vocab_size

    def forward(self, src, src_lengths, tgt, teacher_forcing_ratio=0.5):
        """
        参数：
            src: (batch, src_seq_len) 源序列
            src_lengths: (batch,) 源序列长度
            tgt: (batch, tgt_seq_len) 目标序列
            teacher_forcing_ratio: 使用真实上一步输出的概率
        返回：
            outputs: (batch * tgt_seq_len, vocab_size) 预测
        """
        batch_size = src.size(0)
        tgt_len = tgt.size(1)

        # 编码
        encoder_outputs, (hidden, cell) = self.encoder(src, src_lengths)

        # 创建mask
        max_len = src.size(1)
        mask = torch.arange(max_len, device=src.device).unsqueeze(0) < src_lengths.unsqueeze(1)

        # 解码
        outputs = []
        x = tgt[:, 0:1]  # <start> token

        for t in range(1, tgt_len):
            output, hidden, cell, _ = self.decoder(x, hidden, cell, encoder_outputs, mask)
            outputs.append(output)

            # Teacher forcing
            if self.training and torch.rand(1).item() < teacher_forcing_ratio:
                x = tgt[:, t:t+1]
            else:
                x = output.argmax(dim=1, keepdim=True)

        outputs = torch.stack(outputs, dim=1)  # (batch, tgt_len-1, vocab_size)
        return outputs.view(-1, self.tgt_vocab_size)

    def translate(self, src, src_lengths, max_len=50, start_token=1, end_token=2):
        """推理：翻译源序列"""
        self.eval()
        with torch.no_grad():
            # 编码
            encoder_outputs, (hidden, cell) = self.encoder(src, src_lengths)

            # 创建mask
            max_src_len = src.size(1)
            mask = torch.arange(max_src_len, device=src.device).unsqueeze(0) < src_lengths.unsqueeze(1)

            # 解码
            translations = []
            x = torch.full((src.size(0), 1), start_token, dtype=torch.long, device=src.device)

            for _ in range(max_len):
                output, hidden, cell, _ = self.decoder(x, hidden, cell, encoder_outputs, mask)
                pred = output.argmax(dim=1)

                x = pred.unsqueeze(1)
                translations.append(pred)

            return torch.stack(translations, dim=1)


def train_model(model, train_loader, num_epochs, lr=0.001, device='cuda'):
    """训练函数"""
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(ignore_index=0)  # 忽略padding

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0

        for batch in train_loader:
            src, src_len, tgt = batch
            src = src.to(device)
            src_len = src_len.to(device)
            tgt = tgt.to(device)

            optimizer.zero_grad()

            # 前向传播
            outputs = model(src, src_len, tgt[:, :-1])  # 目标序列去掉最后一个token

            # 计算损失（目标序列去掉第一个token）
            loss = criterion(outputs, tgt[:, 1:].contiguous().view(-1))

            # 反向传播
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # 梯度裁剪
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(train_loader):.4f}")


if __name__ == "__main__":
    # 演示模型结构
    model = Seq2Seq(src_vocab_size=10000, tgt_vocab_size=10000, embed_size=256, hidden_size=512)
    print("Seq2Seq模型结构:")
    print(model)

    # 统计参数量
    total_params = sum(p.numel() for p in model.parameters())
    print(f"\n总参数量: {total_params:,}")


## 练习题

1. **RNN梯度分析**：假设某RNN的权重矩阵 W_hh 的最大特征值为 λ。请分析沿时间反向传播 T 步后，梯度会如何变化（用 λ 表示）。如果 λ > 1，会出现什么问题？

2. **LSTM vs GRU**：比较LSTM和GRU的结构差异。GRU是如何用更少的门来达到类似效果的？在什么情况下应该优先选择LSTM而不是GRU？

3. **序列填充问题**：在处理变长序列时，通常需要对短序列进行填充。请问在计算损失和反向传播时，padding位置是否应该参与计算？应该如何处理？

4. **注意力机制**：假设编码器有5个时刻的输出，隐藏状态维度为256。当前解码器隐藏状态为 h_t。请手工计算使用点积注意力（dot-product attention）时的注意力分数和权重。

5. **双向RNN**：解释双向RNN的工作原理。为什么双向RNN不适用于实时语音识别？它最适合应用于什么场景？

## 延伸阅读

- Hochreiter, S., & Schmidhuber, J. (1997). "Long Short-Term Memory". Neural Computation.
- Cho, K., et al. (2014). "Learning Phrase Representations using RNN Encoder-Decoder for Statistical Machine Translation". EMNLP 2014.
- Bahdanau, D., Cho, K., & Bengio, Y. (2014). "Neural Machine Translation by Jointly Learning to Align and Translate". ICLR 2015.
- Sutskever, I., Vinyals, O., & Le, Q. V. (2014). "Sequence to Sequence Learning with Neural Networks". NIPS 2014.
- Graves, A. (2012). "Supervised Sequence Labelling with Recurrent Neural Networks". Springer.

---
